# Aula 2 — Transformações Avançadas com Spark

**Disciplina:** Big Data Processing — MBA Engenharia de Dados (Mackenzie)

**Objetivo:** Dominar joins, window functions, UDFs e análise de plano de execução.

---

## Instruções

1. Execute cada célula sequencialmente (Shift+Enter)
2. Leia os comentários para entender cada transformação
3. Ao final, resolva o **Desafio** proposto

---

## 1. Configuração e Carga de Dados Multi-Fonte

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("DataFlow-Aula02") \
    .master("local[*]") \
    .config("spark.driver.memory", "2g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.sql.autoBroadcastJoinThreshold", "10m") \
    .getOrCreate()

print(f"SparkSession criada: {spark.version}")

In [ ]:
# Carregar as 3 fontes de dados
from pyspark.sql.functions import explode, col

df_vendas = spark.read.parquet("/home/jovyan/work/data/aula_02/vendas_2023_completo.parquet")
df_clientes = spark.read.parquet("/home/jovyan/work/data/aula_02/clientes.parquet")

# JSON aninhado: precisa explodir o array 'categorias' em linhas
df_categorias_raw = spark.read.json("/home/jovyan/work/data/aula_02/categorias.json", multiLine=True)
df_categorias = df_categorias_raw \
    .select(explode(col("categorias")).alias("cat")) \
    .select(
        col("cat.category_id").alias("category_id"),
        col("cat.category_name").alias("category_name")
    )

print(f"Vendas:     {df_vendas.count():,} registros, {len(df_vendas.columns)} colunas")
print(f"Clientes:   {df_clientes.count():,} registros")
print(f"Categorias: {df_categorias.count():,} registros")
df_categorias.show()

In [ ]:
# Schemas
print("=== VENDAS ===")
df_vendas.printSchema()
print("\n=== CLIENTES ===")
df_clientes.printSchema()
print("\n=== CATEGORIAS ===")
df_categorias.printSchema()

## 2. Joins — Cruzando Fontes de Dados

Vamos enriquecer as vendas com informações de clientes e categorias.

In [ ]:
from pyspark.sql.functions import col, broadcast, coalesce, lit
from pyspark.sql.functions import regexp_extract, concat, lpad, ceil as spark_ceil

# LEFT JOIN com clientes (tabela grande - nao usa broadcast)
df_com_clientes = df_vendas.join(
    df_clientes,
    on="customer_id",
    how="left"
)

# Derivar category_id a partir do product_id
# Logica: PROD_0001 a PROD_0500 -> CAT_01, PROD_0501 a PROD_1000 -> CAT_02, etc.
df_com_clientes = df_com_clientes.withColumn(
    "category_id",
    concat(
        lit("CAT_"),
        lpad(
            spark_ceil(
                regexp_extract(col("product_id"), r"PROD_(\d+)", 1).cast("int") / 500
            ).cast("int").cast("string"),
            2, "0"
        )
    )
)

print(f"Apos join com clientes: {df_com_clientes.count():,} registros")
df_com_clientes.select("order_id", "customer_id", "product_id", "category_id").show(5)

In [ ]:
# BROADCAST JOIN com categorias (tabela pequena - cabe na memoria do Driver)
df_completo = df_com_clientes.join(
    broadcast(df_categorias),
    on="category_id",
    how="left"
)

# Tratar nulls com coalesce
df_completo = df_completo.withColumn(
    "category_name",
    coalesce(col("category_name"), lit("Sem Categoria"))
)

print(f"DataFrame completo: {df_completo.count():,} registros")
df_completo.select("order_id", "customer_id", "category_name", "total_amount").show(5)

In [ ]:
# Verificar no plano de execucao que broadcast foi usado
df_completo.explain(True)

## 3. Window Functions — Rankings e Tendências

Window functions permitem cálculos relativos (ranking, comparação com vizinhos) sem perder linhas.

In [ ]:
from pyspark.sql.functions import sum, count, avg, round, desc
from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank, row_number, lag, lead

# 3.1 Ranking: Top clientes por faturamento em cada estado
# Primeiro, agregar faturamento por cliente+estado
df_faturamento_cliente = df_completo \
    .groupBy("shipping_state", "customer_id") \
    .agg(round(sum("total_amount"), 2).alias("faturamento_total"))

# Definir janela: particionar por estado, ordenar por faturamento desc
window_ranking = Window.partitionBy("shipping_state").orderBy(desc("faturamento_total"))

# Aplicar dense_rank
df_ranking = df_faturamento_cliente.withColumn(
    "rank", dense_rank().over(window_ranking)
)

# Top 3 clientes por estado
print("=== TOP 3 CLIENTES POR ESTADO ===")
df_ranking.filter(col("rank") <= 3).orderBy("shipping_state", "rank").show(15)

In [ ]:
# 3.2 Tendencia: Comparar valor de cada compra com a compra anterior do cliente
from pyspark.sql.functions import to_date

# Janela temporal por cliente, ordenada por data
window_temporal = Window.partitionBy("customer_id").orderBy("order_date")

df_tendencia = df_completo \
    .select("customer_id", "order_date", "total_amount") \
    .withColumn("compra_anterior", lag("total_amount", 1).over(window_temporal)) \
    .withColumn("variacao", 
        round(col("total_amount") - col("compra_anterior"), 2)
    )

print("=== TENDENCIA DE COMPRAS (lag) ===")
df_tendencia.filter(col("compra_anterior").isNotNull()).show(10)

In [ ]:
# 3.3 Running total (soma acumulada) por cliente
window_acumulada = Window.partitionBy("customer_id").orderBy("order_date").rowsBetween(
    Window.unboundedPreceding, Window.currentRow
)

df_acumulado = df_completo \
    .select("customer_id", "order_date", "total_amount") \
    .withColumn("total_acumulado", round(sum("total_amount").over(window_acumulada), 2))

print("=== SOMA ACUMULADA POR CLIENTE ===")
df_acumulado.filter(col("customer_id") == df_acumulado.select("customer_id").first()[0]).show(10)

## 4. UDFs — User Defined Functions

Quando as funções built-in não são suficientes, criamos UDFs customizadas.

In [ ]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

# Definir UDF para classificar ticket
@udf(returnType=StringType())
def classificar_ticket(valor):
    if valor is None:
        return "Indefinido"
    elif valor < 50:
        return "Baixo"
    elif valor < 200:
        return "Medio"
    elif valor < 500:
        return "Alto"
    else:
        return "Premium"

# Aplicar UDF
df_classificado = df_completo.withColumn(
    "faixa_ticket", classificar_ticket(col("total_amount"))
)

# Distribuicao por faixa
df_classificado.groupBy("faixa_ticket") \
    .agg(count("*").alias("qtd"), round(avg("total_amount"), 2).alias("ticket_medio")) \
    .orderBy(desc("qtd")) \
    .show()

### Alternativa sem UDF (mais performática)

UDFs são mais lentas que funções nativas. Sempre prefira `when/otherwise` quando possível.

In [ ]:
from pyspark.sql.functions import when

# Mesma logica sem UDF (mais rapido!)
df_nativo = df_completo.withColumn(
    "faixa_ticket",
    when(col("total_amount") < 50, "Baixo")
    .when(col("total_amount") < 200, "Medio")
    .when(col("total_amount") < 500, "Alto")
    .otherwise("Premium")
)

df_nativo.groupBy("faixa_ticket") \
    .agg(count("*").alias("qtd")) \
    .orderBy(desc("qtd")) \
    .show()

## 5. Plano de Execução (explain)

Entender o plano de execução ajuda a otimizar queries.

In [ ]:
# Comparar plano COM broadcast vs SEM broadcast
# Usando df_com_clientes que ja possui category_id derivado do product_id
print("=== COM BROADCAST (otimizado) ===")
df_com_clientes.join(broadcast(df_categorias), on="category_id", how="left").explain()

print("\n=== SEM BROADCAST (shuffle join) ===")
df_com_clientes.join(df_categorias, on="category_id", how="left").explain()

## 6. Análise Final — Relatório Consolidado

In [ ]:
# Cache do DataFrame completo para reutilizacao
df_completo.cache()

# Faturamento por categoria
print("=== FATURAMENTO POR CATEGORIA ===")
df_completo.groupBy("category_name") \
    .agg(
        round(sum("total_amount"), 2).alias("faturamento"),
        count("order_id").alias("pedidos"),
        round(avg("total_amount"), 2).alias("ticket_medio")
    ) \
    .orderBy(desc("faturamento")) \
    .show()

# Liberar cache
df_completo.unpersist()

In [ ]:
spark.stop()
print("SparkSession encerrada.")

---

# DESAFIO

## Análise de Retenção e Churn de Clientes

Usando os conceitos aprendidos, implemente:

### Requisitos:

1. **Identifique clientes recorrentes** — clientes com mais de 1 compra (use window function `count` over partitionBy customer_id)
2. **Calcule o tempo entre compras** — use `lag` na `order_date` para calcular dias entre compras consecutivas
3. **Classifique o risco de churn:**
   - Ultima compra > 90 dias atras → Alto risco
   - Ultima compra 30-90 dias → Medio risco
   - Ultima compra < 30 dias → Baixo risco
4. **Gere um relatório** com: total de clientes por faixa de risco, ticket médio por faixa
5. **Use `left_anti` join** para encontrar clientes que nunca compraram (orfãos na tabela clientes)

### Dicas:
- Use `datediff(current_date(), max(order_date))` para calcular dias desde última compra
- Use `Window.partitionBy("customer_id")` para calcular métricas por cliente
- Use `df_clientes.join(df_vendas, on="customer_id", how="left_anti")` para encontrar órfãos

### Bonus:
- Compare o `explain()` de um join normal vs broadcast join vs left_anti
- Use `lead()` para prever o próximo valor provável de compra

---

**Boa sorte!** Análise de churn é um caso real em qualquer empresa de e-commerce.

In [ ]:
# ===========================================================
# SEU CODIGO DO DESAFIO AQUI
# ===========================================================

